In [5]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

INPUT_CSV = "ODNRCombined_WellTypes_Joined_Cleaned.csv"
OUTPUT_CSV = "ODNRCombined_WellTypes_Joined_Cleaned_flagged.csv"

LAT_COL = "Well.Latitude"
LON_COL = "Well.Longitude"
TYPE_COL = "Slant"          # 'V' for vertical, 'H' for horizontal
RADIUS_MILES = 7.0          # distance threshold



In [6]:
# ---- 1. Load data ----
df = pd.read_csv(INPUT_CSV)

print("Loaded rows:", len(df))
print("Columns:", df.columns.tolist())

Loaded rows: 169361
Columns: ['Production.Year', 'Quarter..1..2..3..4..n.a.', 'Owner.Name', 'API.Number', 'County', 'Township', 'Oil..Bbls.', 'Gas..Mcf.', 'Brine..Bbls.', 'Days', 'Date.of.First.Production', 'Year', 'Well_Number', 'Well_Name', 'Well_Key', 'Slant', 'Well.Latitude', 'Well.Longitude']


In [7]:
# Ensure we have needed columns
for col in [LAT_COL, LON_COL, TYPE_COL]:
    if col not in df.columns:
        raise ValueError(f"Required column '{col}' not found in CSV.")

# Drop rows without coordinates
coord_mask = df[LAT_COL].notna() & df[LON_COL].notna()
df_valid = df[coord_mask].copy()

print("Rows with valid coordinates:", len(df_valid))


Rows with valid coordinates: 169361


In [8]:
# ---- 2. Identify vertical and horizontal wells ----
slant = df_valid[TYPE_COL].astype(str).str.upper()
is_vertical = slant == "V"
is_horizontal = slant == "H"

vertical = df_valid[is_vertical].copy()
horizontal = df_valid[is_horizontal].copy()

print(f"Vertical wells (valid coords): {len(vertical)}")
print(f"Horizontal wells (valid coords): {len(horizontal)}")

if horizontal.empty:
    raise ValueError("No horizontal wells (Slant == 'H') with valid coords found.")
if vertical.empty:
    raise ValueError("No vertical wells (Slant == 'V') with valid coords found.")

Vertical wells (valid coords): 135254
Horizontal wells (valid coords): 34107


In [9]:
# ---- 3. Build BallTree on horizontal wells using haversine distance ----
# Convert degrees to radians
vert_coords = np.radians(
    np.column_stack([vertical[LAT_COL].values, vertical[LON_COL].values])
)
horiz_coords = np.radians(
    np.column_stack([horizontal[LAT_COL].values, horizontal[LON_COL].values])
)

earth_radius_m = 6371000.0
radius_m = RADIUS_MILES * 1609.34
radius_radians = radius_m / earth_radius_m

tree = BallTree(horiz_coords, metric="haversine")

# For each vertical well, find any horizontals within radius
indices_array = tree.query_radius(vert_coords, r=radius_radians)

near_horizontal = np.array([len(ix) > 0 for ix in indices_array])

In [10]:
# ---- 4. Attach flag back to full dataframe ----
df["VerticalNearHorizontal7mi"] = False  # default

# Map flags back using the original indices of vertical wells
df.loc[vertical.index, "VerticalNearHorizontal7mi"] = near_horizontal

In [11]:
# ---- 5. Save result ----
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved result with flag column to: {OUTPUT_CSV}")

Saved result with flag column to: ODNRCombined_WellTypes_Joined_Cleaned_flagged.csv


In [12]:
import numpy as np

# ---- SANITY CHECK BLOCK ----
N_CHECK = 5   # Number of wells to test

# Filter only vertical wells flagged TRUE
true_verticals = df[(df["Slant"].str.upper() == "V") &
                    (df["VerticalNearHorizontal7mi"]) &
                    df[LAT_COL].notna() & df[LON_COL].notna()]

print("\nSanity check: number of flagged vertical wells:", len(true_verticals))

if len(true_verticals) == 0:
    print("No wells flagged TRUE — check earlier steps.")
else:
    sample = true_verticals.sample(min(N_CHECK, len(true_verticals)), random_state=42)

    print(f"\n=== SANITY CHECK: Showing {len(sample)} randomly-selected vertical wells ===")

    # Convert horizontals to radians (we already did this earlier; reuse arrays)
    # horiz_coords = radians of horizontal wells
    # horizontal = dataframe of horizontal wells

    for idx, row in sample.iterrows():
        vlat = row[LAT_COL]
        vlon = row[LON_COL]
        v_api = row["API.Number"] if "API.Number" in df.columns else idx

        # Convert this vertical well to radians
        v_coord = np.radians([[vlat, vlon]])

        # Query neighbors again
        neighbors = tree.query_radius(v_coord, r=radius_radians, return_distance=True)
        neighbor_indices = neighbors[0][0]
        neighbor_dists_rad = neighbors[1][0]

        print("\n------------------------------------------------------------")
        print(f"VERTICAL WELL API: {v_api}")
        print(f"Location: lat={vlat}, lon={vlon}")
        print(f"Found {len(neighbor_indices)} horizontal wells within 7 miles:")

        # Convert radians to miles
        neighbor_dists_miles = neighbor_dists_rad * earth_radius_m / 1609.34

        # Print each neighbor
        for h_idx, dist in zip(neighbor_indices, neighbor_dists_miles):
            hwell = horizontal.iloc[h_idx]
            h_api = hwell["API.Number"] if "API.Number" in df.columns else h_idx
            print(f"  -> Horizontal API {h_api}, distance: {dist:.2f} miles")

    print("\n=== END SANITY CHECK ===\n")


Sanity check: number of flagged vertical wells: 45752

=== SANITY CHECK: Showing 5 randomly-selected vertical wells ===

------------------------------------------------------------
VERTICAL WELL API: 34111239780000.0
Location: lat=39.699164, lon=-81.127275
Found 965 horizontal wells within 7 miles:
  -> Horizontal API 34111244090000.0, distance: 6.51 miles
  -> Horizontal API 34111244140100.0, distance: 6.52 miles
  -> Horizontal API 34111244090000.0, distance: 6.51 miles
  -> Horizontal API 34111244090000.0, distance: 6.51 miles
  -> Horizontal API 34111244090000.0, distance: 6.51 miles
  -> Horizontal API 34111244090000.0, distance: 6.51 miles
  -> Horizontal API 34111244090000.0, distance: 6.51 miles
  -> Horizontal API 34111244090000.0, distance: 6.51 miles
  -> Horizontal API 34111244090000.0, distance: 6.51 miles
  -> Horizontal API 34111244090000.0, distance: 6.51 miles
  -> Horizontal API 34111244090000.0, distance: 6.51 miles
  -> Horizontal API 34111244090000.0, distance: 6